
 # **Task 1: Binary Classification**
This notebook is for the Flight Delay prediction project. In this notebook the binary classification task is implemented. The two classes are delayed or not. A delay is defined as the plane leaving 15 minutes after the scheduled department time.

**Outline**:
* Baseline
* Logistic Regression
* Random Forest
* Gradient Boosting
* Compare metrics
* Feature importance
* Error breakdown

**To Do**:

* Add visible confusion matrices. -- Better visible results.
*   Write down why we are doing the baseline.
*   Create evaluation metrics comparison table.
*   Decide how to seperate error by reason.


In [ ]:
from google.colab import drive
import numpy as np
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import pandas as pd
import glob

In [ ]:
#------------------------------
# Load data & test train split
#------------------------------

# File paths
file1 = "/content/drive/MyDrive/Junior Year/EE 344/Final Project Data/2014.csv"
file2 = "/content/drive/MyDrive/Junior Year/EE 344/Final Project Data/2015.csv"

# Columns to keep
columns_to_keep = [
    'FL_DATE', 'OP_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'DEP_DELAY'
]

# Define smaller dtypes to save memory
dtypes = {
    'OP_CARRIER': 'category',
    'ORIGIN': 'category',
    'DEST': 'category',
    'CRS_DEP_TIME': 'int32',
    'CRS_ARR_TIME': 'int32',
    'CRS_ELAPSED_TIME': 'float32',
    'DISTANCE': 'float32',
    'CANCELLED': 'int8',
    'DIVERTED': 'int8',
    'DEP_DELAY': 'float32',
    'CANCELLATION_CODE': 'category'
}

# Read CSV files with selected columns and dtypes
df1 = pd.read_csv(file1, usecols=columns_to_keep, dtype=dtypes)
df2 = pd.read_csv(file2, usecols=columns_to_keep, dtype=dtypes)

# Combine into one DataFrame
all_data = pd.concat([df1, df2], ignore_index=True)

# Free memory from intermediate DataFrames
del df1, df2

# Convert date column to datetime
all_data.loc[:, 'FL_DATE'] = pd.to_datetime(all_data['FL_DATE'])

# Remove cancelled/diverted flights to reduce size
all_data = all_data[(all_data['CANCELLED'] == 0) & (all_data['DIVERTED'] == 0)].copy()

# Sort chronologically
all_data = all_data.sort_values("FL_DATE").reset_index(drop=True)

# 70/30 chronological train/test split
split_index = int(len(all_data) * 0.7)
train_df = all_data.iloc[:split_index].copy()
test_df  = all_data.iloc[split_index:].copy()

# Check date ranges
print(train_df["FL_DATE"].min(), "to", train_df["FL_DATE"].max())
print(test_df["FL_DATE"].min(), "to", test_df["FL_DATE"].max())

2014-01-01 00:00:00 to 2015-05-29 00:00:00
2015-05-29 00:00:00 to 2015-12-31 00:00:00


In [ ]:
# ------------------------
# Binary Classification Preprocessing
# ------------------------

def preprocess_flight_data_binary(df):
    df = df.copy()

    # Ensure FL_DATE is datetime
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

    # Extract time features
    df['CRS_DEP_TIME'] = df['CRS_DEP_TIME'].fillna(0).astype(int)
    df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100  # Hour (0–23)

    df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek
    df['MONTH'] = df['FL_DATE'].dt.month

    # Time-of-day category (used for tree models)
    def get_time_of_day(hour):
        if 5 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        else:
            return 'night'

    df['TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

    # Cyclic encoding (used for linear models)
    df['DEP_HOUR_SIN'] = np.sin(2 * np.pi * df['DEP_HOUR'] / 24)
    df['DEP_HOUR_COS'] = np.cos(2 * np.pi * df['DEP_HOUR'] / 24)

    df['DAY_OF_WEEK_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
    df['DAY_OF_WEEK_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)

    return df


# Apply preprocessing
train_data = preprocess_flight_data_binary(train_df)
test_data = preprocess_flight_data_binary(test_df)

print(train_data.head())

     FL_DATE OP_CARRIER ORIGIN DEST  CRS_DEP_TIME  DEP_DELAY  CRS_ARR_TIME  \
0 2014-01-01         AA    ICT  DFW          1135        9.0          1300   
1 2014-01-01         UA    SAN  SFO          1122       18.0          1257   
2 2014-01-01         UA    SFO  LAX          1631       -7.0          1804   
3 2014-01-01         UA    IAH  LAX          1318       25.0          1514   
4 2014-01-01         UA    AUS  DEN           751       -4.0           906   

   CANCELLED CANCELLATION_CODE  DIVERTED  CRS_ELAPSED_TIME  DISTANCE  \
0          0               NaN         0              85.0     328.0   
1          0               NaN         0              95.0     447.0   
2          0               NaN         0              93.0     337.0   
3          0               NaN         0             236.0    1379.0   
4          0               NaN         0             135.0     775.0   

   DEP_HOUR  DAY_OF_WEEK  MONTH TIME_OF_DAY  DEP_HOUR_SIN  DEP_HOUR_COS  \
0        11            

In [ ]:
y_train = (train_data['DEP_DELAY'] > 15).astype(int)
y_test  = (test_data['DEP_DELAY'] > 15).astype(int)

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.sparse import hstack

categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK']

numeric_cols = [
    'DISTANCE',
    'CRS_ELAPSED_TIME',
    'DEP_HOUR_SIN',
    'DEP_HOUR_COS',
    'DAY_OF_WEEK_SIN',
    'DAY_OF_WEEK_COS'
]

# Fit encoder on TRAIN only
ohe = OneHotEncoder(drop='first', sparse_output=True, handle_unknown='ignore')
X_train_cat = ohe.fit_transform(train_data[categorical_cols])
X_test_cat = ohe.transform(test_data[categorical_cols])

# Fit scaler on TRAIN only
scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_data[numeric_cols])
X_test_num = scaler.transform(test_data[numeric_cols])

# Combine
X_train_linear = hstack([X_train_num, X_train_cat], format='csr')
X_test_linear = hstack([X_test_num, X_test_cat], format='csr')

tree_categorical_cols = [
    'OP_CARRIER',
    'ORIGIN',
    'DEST',
    'MONTH',
    'DAY_OF_WEEK',
    'TIME_OF_DAY'
]

tree_numeric_cols = ['DISTANCE', 'CRS_ELAPSED_TIME']

X_train_tree = pd.concat([
    train_data[tree_numeric_cols].astype('float32'),
    train_data[tree_categorical_cols].apply(lambda x: x.astype('category').cat.codes)
], axis=1)

X_test_tree = pd.concat([
    test_data[tree_numeric_cols].astype('float32'),
    test_data[tree_categorical_cols].apply(lambda x: x.astype('category').cat.codes)
], axis=1)

In [ ]:
# ---------------------
# Baseline
# ---------------------
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# Majority baseline
majority_class = y_train.mode()[0]

y_pred_majority = np.full_like(y_test, majority_class)

#Evaluate baseline
print("Majority Baseline Results")
print("F1:", f1_score(y_test, y_pred_majority))
print("Accuracy:", accuracy_score(y_test, y_pred_majority))
print("Precision:", precision_score(y_test, y_pred_majority))
print("Recall:", recall_score(y_test, y_pred_majority))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_majority))

Majority Baseline Results
F1: 0.0
Accuracy: 0.8274587473037549


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Precision: 0.0
Recall: 0.0
Confusion Matrix:
 [[2828019       0]
 [ 589697       0]]


In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'  # handles imbalance
)

log_model.fit(X_train_linear, y_train)

y_pred_log = log_model.predict(X_test_linear)
y_prob_log = log_model.predict_proba(X_test_linear)[:, 1]

print("Logistic Regression Results")
print("F1:", f1_score(y_test, y_pred_log))
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

Logistic Regression Results
F1: 0.3638221927665215
Accuracy: 0.6067952983805559
Precision: 0.25235978388431624
Recall: 0.6516380446229165
ROC-AUC: 0.6696139191798456
Confusion Matrix:
 [[1689585 1138434]
 [ 205428  384269]]


In [ ]:
# After fitting Logistic Regression
del X_train_linear, X_test_linear  # Free up sparse matrices
import gc
gc.collect()  # Force garbage collection

378